# Sentinel-5P / TROPOMI (TROPOspheric Monitoring Instrument)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=sentinel/sentinel-5p-tropomi). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1B

Level 1B: calibrated radiances (RA) and solar irradiances (IR) for the 8 spectral bands.

Searches the Copernicus Data Space catalogue (anonymous). Set DOWNLOAD = True to fetch the first product (several hundred MB).

In [ ]:
%pip install -q requests

In [ ]:
import requests

CATALOGUE = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
AREA = "POLYGON((12.0 41.0, 13.0 41.0, 13.0 42.5, 12.0 42.5, 12.0 41.0))"   # Italy (lon lat)
START, END = "2024-06-01T00:00:00.000Z", "2024-06-02T00:00:00.000Z"

flt = (
    f"Collection/Name eq 'SENTINEL-5P' "
    f"and Attributes/OData.CSC.StringAttribute/any(a:a/Name eq 'productType' "
    f"and a/OData.CSC.StringAttribute/Value eq 'L1B_RA_BD4') "
    f"and ContentDate/Start gt {START} and ContentDate/Start lt {END} "
    f"and OData.CSC.Intersects(area=geography'SRID=4326;{AREA}')"
)
response = requests.get(CATALOGUE, params={"$filter": flt, "$orderby": "ContentDate/Start asc", "$top": 5})
response.raise_for_status()
products = response.json()["value"]
print(len(products), "TROPOMI L1B products found")
for p in products:
    print(p["Name"], round(p["ContentLength"] / 1e6), "MB")

# Searching is anonymous. Downloading needs a free Copernicus Data Space account.
DOWNLOAD = False
if DOWNLOAD and products:
    import getpass
    token = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data={"client_id": "cdse-public", "grant_type": "password",
              "username": input("CDSE e-mail: "), "password": getpass.getpass("CDSE password: ")},
    ).json()["access_token"]
    product = products[0]
    url = "https://download.dataspace.copernicus.eu/odata/v1/Products(%s)/$value" % product["Id"]
    with requests.get(url, headers={"Authorization": "Bearer " + token}, stream=True) as r:
        r.raise_for_status()
        with open(product["Name"] + ".zip", "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print("saved", product["Name"] + ".zip")

## L2

Level 2: tropospheric/total columns of NO2, O3, SO2, CO, CH4, HCHO plus aerosol index, aerosol layer height and cloud properties.

Downloads one TROPOMI offline NO2 orbit from the Planetary Computer (no account needed; several hundred MB) and maps the quality-filtered tropospheric column.

In [ ]:
%pip install -q pystac-client planetary-computer xarray netCDF4 requests matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import pystac_client
import requests
import xarray as xr

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)



search = catalog.search(
    collections=["sentinel-5p-l2-netcdf"],
    bbox=[12.4, 41.8, 12.6, 42.0],                      # Rome - change to your area
    datetime="2024-06-01/2024-06-03",
    query={"s5p:processing_mode": {"eq": "OFFL"}},
)
items = [i for i in search.item_collection() if "no2" in i.assets]
if not items:
    raise SystemExit("No NO2 products found - widen the dates or the bounding box")
item = items[0]
print(len(items), "NO2 products; using", item.id)

# One orbit is a few hundred MB
with requests.get(item.assets["no2"].href, stream=True) as r:
    r.raise_for_status()
    with open("no2.nc", "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)

ds = xr.open_dataset("no2.nc", group="PRODUCT")
no2 = ds["nitrogendioxide_tropospheric_column"][0]
qa = ds["qa_value"][0]
no2 = no2.where(qa > 0.75)                   # recommended quality filter
plt.pcolormesh(ds["longitude"][0], ds["latitude"][0], no2, vmin=0, vmax=2e-4, cmap="magma")
plt.colorbar(label="tropospheric NO2 column (mol/m2)")
plt.title(item.id)
plt.show()